# Matching Exercises for Greek Protipa Exams



In [1]:
%load_ext autoreload
%autoreload 2

# This allows live-editing of the source code in 'src/protipa_exams_dataset/' 
# without having to restart the kernel. Changes to .py files (like data_loader.py) 
# will be automatically detected and reloaded whenever any cell is executed.

In [5]:
from pathlib import Path
import sys

project_root = Path.cwd().parent 
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    print(f"✅ Προστέθηκε το {src_path} στο path!")

✅ Προστέθηκε το c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά\src στο path!


In [6]:
import json
import logging
import requests
import os
import random
import time
import traceback

import lm_eval
import pandas as pd
import yaml
from datasets import load_dataset, concatenate_datasets
from dotenv import load_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager
from protipa_exams_dataset.data_loader import load_protipa_dataset, clean_dataset_paths, apply_matching_processing, clean_dataset_paths

import warnings
# Silence the "SyntaxWarning" specifically for invalid escape sequences 
# that are present in the raw dataset strings.
warnings.filterwarnings('ignore', category=SyntaxWarning, message='invalid escape sequence')



import IPython.display
import random
import pandas as pd
random.seed(42)
# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [7]:
# Load the full dataset (concatenates all splits by default)
dataset = load_protipa_dataset()

# Convert to a pandas DataFrame
df = dataset.to_pandas()

# Set a character limit for column display (e.g., 50 characters)
pd.set_option('display.max_colwidth', 500)

columns_to_show = ['unique_id', 'question', 'input', 'choices', 'answer', 'answer_index']

# Display the results
df[columns_to_show].head()

2026-01-15 21:46:55 - INFO - Loading dataset from Hugging Face: PennyK98/protipa_exams_dataset


,unique_id,question,input,choices,answer,answer_index
0,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2020_8_12,Να συμπληρώσετε σωστά τη φράση: Ο κινηματογραφιστής αφηγήθηκε στον βοηθό του __________.,,"[Α. ό,τι συνέβη., Β. ό,τι συνέβει., Γ. ότι συνέβη., Δ. ότι συνέβει.]",Α,0
1,ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2021_1_4,"Παντοπώλης είχε ένα γεμάτο τσουβάλι με ρύζι. Την πρώτη μέρα που το άνοιξε πούλησε το $\frac{1}{3}$, τη δεύτερη πούλησε το $\frac{1}{4}$ από το ρύζι που του έμεινε, την τρίτη μέρα τα $\frac{2}{3}$ από το ρύζι που του είχε μείνει και τελικά του έμειναν 3 κιλά. Πόσα κιλά ρύζι είχε το γεμάτο τσουβάλι;",,"[A. 9, B. 12, Γ. 18, Δ. 21]",Γ,2
2,ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2021_2_41,"Ποια είναι η διάμεσος των αριθμών: $\sqrt{3}, 8$;",,"[A. 8,5, B. 9, Γ. 10, Δ. 8]",Α,0
3,ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΓΥΜΝΑΣΙΟ_2020_2_4,Η παράσταση $\frac{2}{5}\cdot(4-\frac{3}{7})$ είναι ίση με:,,"[A. $\frac{10}{7}$, B. $\frac{1}{2}$, Γ. 7, Δ. 4]",Α,0
4,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2019_4,"Σύμφωνα με το κείμενο Β, να χαρακτηρίσετε ως **Σωστό (Σ)** ή **Λάθος (Λ)** το περιεχόμενο της πρότασης: Ο Γιάννης Τροχόπουλος υπήρξε στο παρελθόν μέλος του Διοικητικού Συμβουλίου της Δημόσιας Κεντρικής Βιβλιοθήκης της Βέροιας.","Kείμενο Β: Τάμπλετ φορτωμένα με εφημερίδες στην αραβική γλώσσα για τους πρόσφυγες! \nΔεν παύει να μας εκπλήσσει η εφευρετικότητα και η ταχύτητα με την οποία οργανώνει τις υπηρεσίες της η Δημόσια Κεντρική Βιβλιοθήκη της Βέροιας. Διεισδύει στην κοινωνία, αφουγκράζεται τις ανάγκες της, καινοτομεί, πειραματίζεται, οραματίζεται τη βιβλιοθήκη του μέλλοντος. […] Οι λέξεις είναι λίγες για να περιγράψουν τις εικόνες, το ετερόκλητο πλήθος, τις υπηρεσίες που συνυπάρχουν στους ορόφους της. Ανάμεσά τους ...","[α. Σωστό, β. Λάθος]",Λάθος,1


In [61]:
target_ids = [
    "ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΓΥΜΝΑΣΙΟ_2016_7",
    "ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2016_2_7",
    "ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΓΥΜΝΑΣΙΟ_2017_3_6",
    "ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2019_8"
]


df[df['unique_id'].isin(target_ids)][columns_to_show].head()

,unique_id,question,input,choices,answer,answer_index
184,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΓΥΜΝΑΣΙΟ_2017_3_6,Αντιστοιχίστε τις λέξεις της αριστερής στήλης με τις λέξεις της δεξιάς στήλης. Να καταγράψετε τις απαντήσεις σας στον πίνακα που ακολουθεί συμπληρώνοντας τα μικρά γράμματα δίπλα στα κεφαλαία.,"Κείμενο: Μία μορφή ψυχαγωγίας, με περιεχόμενο όμως θρησκευτικό και διδακτικό, ήταν οι θεατρικές παραστάσεις. Στην Αθήνα, κατά τα «Λήναια» και τα «Μεγάλα Διονύσια», το κοινό παρακολουθούσε, εγκατεστημένο στο θέατρο από την ανατολή του ήλιου, δραματικά έργα. Το εισιτήριο στοίχιζε δύο οβολούς αλλά για τους άπορους κάλυπτε η πολιτεία τις δαπάνες. Τα θεάματα παρακολουθούσαν και γυναίκες. Επειδή οι παραστάσεις διαρκούσαν ολόκληρη μέρα, οι θεατές έπαιρναν μαζί τους φαγητά. Το κοινό επιδοκίμαζε ή απ...","[Α. θέατρο, Β. χορός, Γ. πανηγύρια, α. συναναστροφή, β. λατρεία, γ. διασκέδαση, δ. εύθυμη διάθεση, ε. καλή σωματική κατάσταση, στ. καλλιέργεια πνεύματος]","[""Α-β"", ""Α-γ"", ""Β-ε"", ""Β-στ"", ""Γ-α"", ""Γ-β"", ""Γ-δ""]",None
1514,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2019_8,Να αντιστοιχίσετε τα σημεία στίξης (Στήλη Α) με αυτό που δηλώνουν (Στήλη Β):,,"[1. Τάμπλετ φορτωμένα με εφημερίδες στην αραβική γλώσσα για τους πρόσφυγες! (θαυμαστικό), 2. Το παράρτημα της βιβλιοθήκης στο σχολείο της (11ο Δημοτικό) (παρένθεση), 3. η μικρή «καταβρόχθιζε» λογοτεχνία (εισαγωγικά), 4. Πώς φαντάζεται μια κοινωνία χωρίς βιβλιοθήκη; (ερωτηματικό), 5. «Πολύ μικρός διάβαζα περισσότερο. Τώρα λόγω του σχολείου…λογοτεχνικό βιβλίο» (εισαγωγικά), α. επεξήγηση, β. ερώτηση, γ. θαυμασμό, δ. λόγια προσώπου που αναφέρονται κατά λέξη-ευθύς λόγος, ε. μεταφορικό λόγο]","[""1-γ"", ""2-α"", ""3-ε"", ""4-β"", ""5-δ""]",None
1520,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2016_2_7,Να αντιστοιχίσετε την κάθε φράση από την αριστερή στήλη (α-στ) με την ορθή επεξήγηση από τη δεξιά στήλη (1-12) και να γράψετε τον κατάλληλο αριθμό. Προσοχή! Οι ορθές απαντήσεις είναι πέντε (5).,,"[α. το έξω από το φυσικό, β. κατανομή του πληθυσμού, γ. ακμαία κέντρα, δ. πόροι ζωής, ε. γεωργικοί πληθυσμοί, στ. ορεινό συγκρότημα, 1. άνθρωποι που ασχολούνται με τα φυτά και καλλιεργούν τη γη, 2. σύμπλεγμα περιοχών σε μεγάλο υψόμετρο που μπορούν να κατοικηθούν, 3. η ύπαρξη λίγων πόλεων και πολλών χωριών., 4. το αντίθετο από αυτό που γνωρίζουν ή περιμένουν οι άνθρωποι., 5. το ποσοστό του πληθυσμού που κατοικεί σε πόλεις και χωριά, 6. τα υλικά αγαθά της ζωής, 7. πόλεις με μεγάλο πληθυσμό και...","[""β-5"", ""γ-7"", ""δ-12"", ""ε-11"", ""στ-2""]",None
1628,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΓΥΜΝΑΣΙΟ_2016_7,"Γράψε τις φράσεις (1-5) στη στήλη (Α-Γ) στην οποία ταιριάζει η καθεμιά, σύμφωνα με το κείμενο 2:","Κείμενο 2: Η Οικογένεια Μπελιέ. \nΗ Πολά είναι η μοναδική φωνή στην τετραμελή οικογένειά της. Ο πατέρας της, η μητέρα της και ο αδερφός της είναι όλοι κωφάλαλοι και καθημερινά αναλαμβάνει τον ρόλο του διερμηνέα για να τους βοηθάει. Φροντίζει ενεργά τη διαχείριση της φάρμας τους, πουλάει με κέφι τα γαλακτοκομικά προϊόντα τους και, παράλληλα, πηγαίνει στο σχολείο. Εκεί, ο καθηγητής της μουσικής αντιλαμβάνεται τις δυνατότητες της φωνής της, την αναγκάζει να γίνει μέλος της σχολικής χορωδίας και...","[Α. Οικογενειακή ζωή, Β. Σχολική δραστηριότητα, Γ. Μελλοντική σταδιοδρομία, 1. αναλαμβάνει τον ρόλο του διερμηνέα, 2. Φροντίζει ενεργά τη διαχείριση της φάρμας, 3. την αναγκάζει να γίνει μέλος της σχολικής χορωδίας, 4. λαμπρή καριέρα στο τραγούδι, 5. δε νιώθει έτοιμη να εγκαταλείψει τους γονείς της]","[""Α-1"", ""Α-2"", ""Α-5"", ""Β-3"", ""Γ-4""]",None


In [9]:
# Pass the IDs directly to the function
matching_questions = df[df['exercise_type'] == 'Matching']
target_ids = matching_questions['unique_id'].tolist()
df_final = apply_matching_processing(df, target_ids=target_ids)
# Filter the resulting dataframe to see those IDs
df_final[df_final['unique_id'].isin(target_ids)][columns_to_show]

,unique_id,question,input,choices,answer,answer_index
184,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΓΥΜΝΑΣΙΟ_2017_3_6,Αντιστοιχίστε τις λέξεις της αριστερής στήλης με τις λέξεις της δεξιάς στήλης. Να καταγράψετε τις απαντήσεις σας στον πίνακα που ακολουθεί συμπληρώνοντας τα μικρά γράμματα δίπλα στα κεφαλαία.,"Κείμενο: Μία μορφή ψυχαγωγίας, με περιεχόμενο όμως θρησκευτικό και διδακτικό, ήταν οι θεατρικές παραστάσεις. Στην Αθήνα, κατά τα «Λήναια» και τα «Μεγάλα Διονύσια», το κοινό παρακολουθούσε, εγκατεστημένο στο θέατρο από την ανατολή του ήλιου, δραματικά έργα. Το εισιτήριο στοίχιζε δύο οβολούς αλλά για τους άπορους κάλυπτε η πολιτεία τις δαπάνες. Τα θεάματα παρακολουθούσαν και γυναίκες. Επειδή οι παραστάσεις διαρκούσαν ολόκληρη μέρα, οι θεατές έπαιρναν μαζί τους φαγητά. Το κοινό επιδοκίμαζε ή απ...","[Α. θέατρο, Β. χορός, Γ. πανηγύρια, α. συναναστροφή, β. λατρεία, γ. διασκέδαση, δ. εύθυμη διάθεση, ε. καλή σωματική κατάσταση, στ. καλλιέργεια πνεύματος]","[[Α-β, Α-β, Β-α, Β-δ, Γ-ε, Γ-γ, Γ-στ], [Α-β, Α-β, Β-δ, Β-ε, Γ-α, Γ-στ, Γ-γ], [Α-β, Α-γ, Β-ε, Β-στ, Γ-α, Γ-β, Γ-δ], [Α-β, Α-δ, Β-γ, Β-α, Γ-ε, Γ-β, Γ-στ]]",2
1514,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2019_8,Να αντιστοιχίσετε τα σημεία στίξης (Στήλη Α) με αυτό που δηλώνουν (Στήλη Β):,,"[1. Τάμπλετ φορτωμένα με εφημερίδες στην αραβική γλώσσα για τους πρόσφυγες! (θαυμαστικό), 2. Το παράρτημα της βιβλιοθήκης στο σχολείο της (11ο Δημοτικό) (παρένθεση), 3. η μικρή «καταβρόχθιζε» λογοτεχνία (εισαγωγικά), 4. Πώς φαντάζεται μια κοινωνία χωρίς βιβλιοθήκη; (ερωτηματικό), 5. «Πολύ μικρός διάβαζα περισσότερο. Τώρα λόγω του σχολείου…λογοτεχνικό βιβλίο» (εισαγωγικά), α. επεξήγηση, β. ερώτηση, γ. θαυμασμό, δ. λόγια προσώπου που αναφέρονται κατά λέξη-ευθύς λόγος, ε. μεταφορικό λόγο]","[[1-γ, 2-α, 3-ε, 4-β, 5-δ], [1-δ, 2-ε, 3-γ, 4-β, 5-α], [1-γ, 2-δ, 3-α, 4-β, 5-ε], [1-ε, 2-δ, 3-γ, 4-β, 5-α]]",0
1520,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2016_2_7,Να αντιστοιχίσετε την κάθε φράση από την αριστερή στήλη (α-στ) με την ορθή επεξήγηση από τη δεξιά στήλη (1-12) και να γράψετε τον κατάλληλο αριθμό. Προσοχή! Οι ορθές απαντήσεις είναι πέντε (5).,,"[α. το έξω από το φυσικό, β. κατανομή του πληθυσμού, γ. ακμαία κέντρα, δ. πόροι ζωής, ε. γεωργικοί πληθυσμοί, στ. ορεινό συγκρότημα, 1. άνθρωποι που ασχολούνται με τα φυτά και καλλιεργούν τη γη, 2. σύμπλεγμα περιοχών σε μεγάλο υψόμετρο που μπορούν να κατοικηθούν, 3. η ύπαρξη λίγων πόλεων και πολλών χωριών., 4. το αντίθετο από αυτό που γνωρίζουν ή περιμένουν οι άνθρωποι., 5. το ποσοστό του πληθυσμού που κατοικεί σε πόλεις και χωριά, 6. τα υλικά αγαθά της ζωής, 7. πόλεις με μεγάλο πληθυσμό και...","[[β-5, γ-11, δ-12, ε-2, στ-7], [β-5, γ-12, δ-11, ε-7, στ-2], [β-5, γ-2, δ-12, ε-11, στ-7], [β-5, γ-7, δ-12, ε-11, στ-2]]",3
1628,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΓΥΜΝΑΣΙΟ_2016_7,"Γράψε τις φράσεις (1-5) στη στήλη (Α-Γ) στην οποία ταιριάζει η καθεμιά, σύμφωνα με το κείμενο 2:","Κείμενο 2: Η Οικογένεια Μπελιέ. \nΗ Πολά είναι η μοναδική φωνή στην τετραμελή οικογένειά της. Ο πατέρας της, η μητέρα της και ο αδερφός της είναι όλοι κωφάλαλοι και καθημερινά αναλαμβάνει τον ρόλο του διερμηνέα για να τους βοηθάει. Φροντίζει ενεργά τη διαχείριση της φάρμας τους, πουλάει με κέφι τα γαλακτοκομικά προϊόντα τους και, παράλληλα, πηγαίνει στο σχολείο. Εκεί, ο καθηγητής της μουσικής αντιλαμβάνεται τις δυνατότητες της φωνής της, την αναγκάζει να γίνει μέλος της σχολικής χορωδίας και...","[Α. Οικογενειακή ζωή, Β. Σχολική δραστηριότητα, Γ. Μελλοντική σταδιοδρομία, 1. αναλαμβάνει τον ρόλο του διερμηνέα, 2. Φροντίζει ενεργά τη διαχείριση της φάρμας, 3. την αναγκάζει να γίνει μέλος της σχολικής χορωδίας, 4. λαμπρή καριέρα στο τραγούδι, 5. δε νιώθει έτοιμη να εγκαταλείψει τους γονείς της]","[[Α-1, Α-2, Α-5, Β-3, Γ-4], [Α-5, Α-4, Α-1, Β-2, Γ-3], [Α-3, Α-5, Α-2, Β-4, Γ-1], [Α-4, Α-2, Α-5, Β-1, Γ-3]]",0


In [10]:
df_final[columns_to_show].head()

,unique_id,question,input,choices,answer,answer_index
0,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2020_8_12,Να συμπληρώσετε σωστά τη φράση: Ο κινηματογραφιστής αφηγήθηκε στον βοηθό του __________.,,"[Α. ό,τι συνέβη., Β. ό,τι συνέβει., Γ. ότι συνέβη., Δ. ότι συνέβει.]",Α,0
1,ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2021_1_4,"Παντοπώλης είχε ένα γεμάτο τσουβάλι με ρύζι. Την πρώτη μέρα που το άνοιξε πούλησε το $\frac{1}{3}$, τη δεύτερη πούλησε το $\frac{1}{4}$ από το ρύζι που του έμεινε, την τρίτη μέρα τα $\frac{2}{3}$ από το ρύζι που του είχε μείνει και τελικά του έμειναν 3 κιλά. Πόσα κιλά ρύζι είχε το γεμάτο τσουβάλι;",,"[A. 9, B. 12, Γ. 18, Δ. 21]",Γ,2
2,ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΛΥΚΕΙΟ_2021_2_41,"Ποια είναι η διάμεσος των αριθμών: $\sqrt{3}, 8$;",,"[A. 8,5, B. 9, Γ. 10, Δ. 8]",Α,0
3,ΘΕΜΑΤΑ_ΜΑΘΗΜΑΤΙΚΑ_ΓΥΜΝΑΣΙΟ_2020_2_4,Η παράσταση $\frac{2}{5}\cdot(4-\frac{3}{7})$ είναι ίση με:,,"[A. $\frac{10}{7}$, B. $\frac{1}{2}$, Γ. 7, Δ. 4]",Α,0
4,ΘΕΜΑΤΑ_ΓΛΩΣΣΑ_ΛΥΚΕΙΟ_2019_4,"Σύμφωνα με το κείμενο Β, να χαρακτηρίσετε ως **Σωστό (Σ)** ή **Λάθος (Λ)** το περιεχόμενο της πρότασης: Ο Γιάννης Τροχόπουλος υπήρξε στο παρελθόν μέλος του Διοικητικού Συμβουλίου της Δημόσιας Κεντρικής Βιβλιοθήκης της Βέροιας.","Kείμενο Β: Τάμπλετ φορτωμένα με εφημερίδες στην αραβική γλώσσα για τους πρόσφυγες! \nΔεν παύει να μας εκπλήσσει η εφευρετικότητα και η ταχύτητα με την οποία οργανώνει τις υπηρεσίες της η Δημόσια Κεντρική Βιβλιοθήκη της Βέροιας. Διεισδύει στην κοινωνία, αφουγκράζεται τις ανάγκες της, καινοτομεί, πειραματίζεται, οραματίζεται τη βιβλιοθήκη του μέλλοντος. […] Οι λέξεις είναι λίγες για να περιγράψουν τις εικόνες, το ετερόκλητο πλήθος, τις υπηρεσίες που συνυπάρχουν στους ορόφους της. Ανάμεσά τους ...","[α. Σωστό, β. Λάθος]",Λάθος,1
